# Day 1: I Own the Training Loop

**Goal**: Train an NLP model from scratch and debug it confidently.

**Time**: 4 hours
- 20 min: Planning
- 2.5 hrs: Coding
- 45 min: Debugging & experiments
- 25 min: Reflection

---

## Confidence Checks
- [ ] I can explain why incorrect padding or masking breaks training
- [ ] I can change embedding size or max sequence length without panic
- [ ] I know exactly where NaNs come from

## Task 1: Text Pipeline (45 min)

**Dataset**: IMDb (5k samples)

Build:
- Whitespace tokenizer
- Vocabulary (PAD=0, UNK=1)
- Padding and attention masks
- Custom Dataset and DataLoader

In [195]:
# Imports
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchmetrics
import torch.nn.functional as F
# Import everything from nlp_utils package
from nlp_utils import (
    tokenize,
    Vocabulary,
    ReviewDataSet,
    collate_fn,
    extract_imdb_sample_as_dict,
    SentimentModel,
    Trainer
)

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [20]:
from datasets import load_dataset

In [21]:
ds = load_dataset("stanfordnlp/imdb")

In [22]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [23]:
# Extract sampled splits as dictionaries
train, test = extract_imdb_sample_as_dict(ds, train_size=5000, test_size=1000)

In [24]:
type(train)

dict

In [25]:
sample_text = "Hello! This is a simple example. Let's tokenize this text."
    
tokens = tokenize(sample_text)

In [26]:
# Create and build vocabulary
vocab = Vocabulary(tokenizer=tokenize)
vocab.build_from_texts(train['text'])

Vocabulary size: 38553


In [27]:
# Create datasets
train_dataset = ReviewDataSet(target=train, vocab=vocab)
test_dataset = ReviewDataSet(target=test, vocab=vocab)

In [28]:
# Create DataLoaders
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,          # Shuffle for training
    collate_fn=collate_fn,
    num_workers=0          # Set to 0 for debugging, increase for speed
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,         # Don't shuffle for testing
    collate_fn=collate_fn,
    num_workers=0
)




In [29]:
batch=next(iter(train_loader))

In [30]:
vocab.get_vocab_size()

38553

In [31]:
vocab.pad_token_id

0

# Training

## Task 2: Training Loop (90 min)

**Build from scratch with validation after every epoch**

In [32]:
# Model is now in nlp_utils package!
# No need to define it here - just import

In [33]:
model=SentimentModel(vocab=vocab)
output=model(batch)

In [34]:
output.shape

torch.Size([32, 2])

In [36]:
# Initialize model and optimizer
model = SentimentModel(vocab=vocab, embedding_dim=256, hidden_dim=128, num_categories=2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

# Define metrics using torchmetrics
metrics = {
    'acc': torchmetrics.Accuracy(task='multiclass', num_classes=2),
    'f1': torchmetrics.F1Score(task='multiclass', num_classes=2)
}

# Create trainer
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    device=device,
    metrics=metrics,
    grad_clip_norm=1.0
)

# Train!
history = trainer.fit(num_epochs=5)


Epoch 1/5


Validation: 100%|██████████| 32/32 [00:00<00:00, 285.13it/s]



Results:
  Train - Loss: 0.6378 | acc: 0.6256 | f1: 0.6256
  Val   - Loss: 0.5553 | acc: 0.7260 | f1: 0.7260
  ✓ New best validation loss!

Epoch 2/5


Validation: 100%|██████████| 32/32 [00:00<00:00, 256.32it/s]



Results:
  Train - Loss: 0.4598 | acc: 0.7896 | f1: 0.7896
  Val   - Loss: 0.4421 | acc: 0.7950 | f1: 0.7950
  ✓ New best validation loss!

Epoch 3/5


...


Results:
  Train - Loss: 0.3273 | acc: 0.8620 | f1: 0.8620
  Val   - Loss: 0.4152 | acc: 0.8140 | f1: 0.8140
  ✓ New best validation loss!

Epoch 4/5


Validation: 100%|██████████| 32/32 [00:00<00:00, 282.34it/s]



Results:
  Train - Loss: 0.2532 | acc: 0.8966 | f1: 0.8966
  Val   - Loss: 0.3733 | acc: 0.8350 | f1: 0.8350
  ✓ New best validation loss!

Epoch 5/5


Validation: 100%|██████████| 32/32 [00:00<00:00, 271.78it/s]


Results:
  Train - Loss: 0.1900 | acc: 0.9282 | f1: 0.9282
  Val   - Loss: 0.3802 | acc: 0.8400 | f1: 0.8400


In [183]:
def cross_entropy_loss(logits,targets):
    #check if logits have batch dimention otherwise add
    N=len(targets)
    n_c=logits.shape[-1]
    if logits.shape[0] !=N:
        raise ValueError(f"Unmatched shape between the two logits={targets.shape} targets={logits.shape}")
   
    exps=torch.exp(logits)
    prob=exps/exps.sum(dim=-1,keepdim=True)
    #for gather to work it needs same dimension so batch_size to batch_sizeX1
    y_extended_dim=y.unsqueeze(-1)
    selection=prob.gather(-1,y_extended_dim)
    loss=-torch.log(selection)
    return loss.mean()
    
    
    

In [184]:
#lets do computation for one batch
def run_model_for_batch(batch,model,device=device,loss_fn=nn.CrossEntropyLoss()):
    model.eval()
    model.to(device)
    device_mapped_batch={k:v.to(device) for  k,v in batch.items()}
    with torch.no_grad():
        
        result= model(device_mapped_batch)
        y=device_mapped_batch["label"]
        loss=loss_fn(result,y)
    return result,y,loss
        
        
        

In [185]:
result,y,ce_loss=run_model_for_batch(model=model,batch=batch)
result,y,my_loss=run_model_for_batch(model=model,batch=batch,loss_fn=cross_entropy_loss)


In [191]:

abs(ce_loss - my_loss) < 1e-8 

tensor(True, device='cuda:0')

In [187]:
ce_loss

tensor(0.0606, device='cuda:0')

In [188]:
my_loss

tensor(0.0606, device='cuda:0')

# Scenario 1: Exploding gradients from high learning rate

In [194]:

print("=== Scenario 1: High Learning Rate ===\n")

# Create fresh model
bad_model = SentimentModel(vocab=vocab, embedding_dim=256, hidden_dim=128, num_categories=2)

# INTENTIONALLY BAD: Learning rate = 1.0 (way too high!)
bad_optimizer = torch.optim.Adam(bad_model.parameters(), lr=1e10)

# Train for just a few batches
bad_model.to(device)
bad_model.train()

for i, batch in enumerate(train_loader):
    if i >= 5:  # Only 5 batches
        break
        
    batch = {k: v.to(device) for k, v in batch.items()}
    bad_optimizer.zero_grad()
    
    logits = bad_model(batch)
    loss = nn.CrossEntropyLoss()(logits, batch['label'])
    
    print(f"Batch {i}: Loss = {loss.item():.4f}")
    
    # Check for NaN
    if torch.isnan(loss):
        print(f"❌ NaN detected at batch {i}!")
        break
        
    loss.backward()
    bad_optimizer.step()

=== Scenario 1: High Learning Rate ===

Batch 0: Loss = 0.6925
Batch 1: Loss = 49798314146729727890331743027200.0000
Batch 2: Loss = 1208340583461057011181217439023104.0000
Batch 3: Loss = 561309790091622882359180972261376.0000
Batch 4: Loss = 236265266659004779631727186280448.0000
